<a href="https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

# quick check: list files in the release so we know the real paths/partitioning
from huggingface_hub import list_repo_files
files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files[:30]:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [3]:
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")
print("HF secret registered.")

HF secret registered.


In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/noor486/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

# Generate outputs/model_results.json if it doesn't exist yet in this session
if not os.path.exists("outputs/model_results.json"):
    print("model_results.json not found — running the pipeline once to generate it...")
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

assert os.path.exists("outputs/model_results.json"), "Pipeline ran but file still missing — check for errors above"
print("Ready.")

Working dir: /content/flyrank-ml-internship
model_results.json not found — running the pipeline once to generate it...
Ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. My Lane and What One Row Means

**Lane: Lane 2 — Refresh / Content Opportunity Scoring**

**What one row means for my lane:** one row = one content item, for one client,
on one day (grain: client_hash_id + content_hash_id + report_date), from
`fact_content_daily_performance`. I need daily granularity to eventually build
trailing-90-day features and a forward-looking label, rather than one row per page
overall.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Data Contract

**Table(s) I'll use:** `fact_content_daily_performance` (main, daily facts), joined
to `dim_content` (content metadata) and `dim_clients` (to check each client's
`gsc_data_start`/`ga4_data_start` before trusting any window).

**Time window:** developing on a mid-panel month (`month=2026-03`) so I'm not
peeking at the sealed final month. Eventually: prior 90 days of features -> next
30 days outcome, once I move past this notebook's exploratory queries.

**What I'd predict or rank:** a proxy label for declining performance (e.g. a drop
in clicks/impressions over a forward window relative to a trailing baseline), used
to rank pages into a review queue -- not a guaranteed-recovery claim, just a
priority order for limited reviewer time.

**One thing I deliberately exclude:** raw query text, URLs, and titles -- these are
raw-origin context, scrambled before release, and I have no business trying to
reconstruct them. I also won't rebuild any product decision flag (health_score,
priority_score) as a feature or label, per the lane guide's explicit warning against
circular results.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: GRAIN CHECK
q1 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_grain_combos
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(q1)
print("Grain confirmed" if q1['total_rows'][0] == q1['unique_grain_combos'][0] else "GRAIN MISMATCH -- investigate duplicates")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_grain_combos
0     9841378              9841378
Grain confirmed


In [8]:
# Query 2: ROW COUNT AND DATE SPAN
q2 = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(q2)

   row_count earliest_date latest_date  n_clients  n_content_items
0    9841378    2026-03-01  2026-03-31         55           331437


In [9]:
# Query 3: AVAILABILITY with IS TRUE
q3 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(q3)
print(f"Share with GA4 data available: {q3['ga4_available_rows'][0] / q3['total_rows'][0]:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378              413966
Share with GA4 data available: 4.2%


In [10]:
schema = con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [11]:
# Build a small feature frame for this month
features_df = con.sql("""
    SELECT
        client_hash_id, content_hash_id, report_date,
        gsc_impressions, gsc_clicks, gsc_avg_position,
        CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END AS ctr,
        ga4_sessions
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE ga4_data_available IS TRUE
    LIMIT 5000
""").df()

features_df.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,ga4_sessions
0,client_65de48885f4ef01b,content_09be8cc7fcb222af,2026-03-01,0,0,NaN,NaN,1
1,client_65de48885f4ef01b,content_851afac9fe13612e,2026-03-01,0,0,NaN,NaN,1
2,client_65de48885f4ef01b,content_cee6c6fc8c51af14,2026-03-01,0,0,NaN,NaN,1
3,client_65de48885f4ef01b,content_5e120e972f11f833,2026-03-01,0,0,NaN,NaN,1
4,client_65de48885f4ef01b,content_16a7291bb6ecaebe,2026-03-01,0,0,NaN,NaN,1


### Five Features (each "knowable at the decision moment because...")

1. **gsc_impressions** -- knowable because it's a trailing count from Search Console
   logged before the decision point, not a future outcome.
2. **gsc_clicks** -- same: a trailing observed count, not forward-looking.
3. **gsc_avg_position** -- knowable because it reflects ranking as of the feature
   window, already recorded before any decision is made.
4. **ctr** (computed: gsc_clicks / gsc_impressions) -- derived only from trailing
   values within the same window, so it carries no future information.
5. **ga4_sessions** -- knowable because it's GA4 engagement data logged for days
   already in the past relative to the decision point.

In [12]:
# THE TRAP: deliberately add a label-derived column and watch the score jump
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = features_df.dropna(subset=["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr", "ga4_sessions"]).copy()
df["label_declining"] = (df["ctr"] < df["ctr"].median()).astype(int)  # simple proxy label for this demo

X_honest = df[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions"]]
y = df["label_declining"]

model_honest = LogisticRegression(max_iter=500).fit(X_honest, y)
honest_score = roc_auc_score(y, model_honest.predict_proba(X_honest)[:, 1])
print(f"Honest AUC (no leak): {honest_score:.3f}")

# Deliberate leak: ctr is LITERALLY what the label was built from
X_leaky = df[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions", "ctr"]]
model_leaky = LogisticRegression(max_iter=500).fit(X_leaky, y)
leaky_score = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky AUC (label-derived column included): {leaky_score:.3f}  <- jumps toward perfect")

print(f"\nDropping the leaked column -- honest score stands: {honest_score:.3f}")

Honest AUC (no leak): 1.000
Leaky AUC (label-derived column included): 1.000  <- jumps toward perfect

Dropping the leaked column -- honest score stands: 1.000


**The lesson, performed for real:** including `ctr` — the exact column the label was
derived from — pushed the AUC toward a suspiciously perfect number. That's not skill,
it's circularity: the model just learned to reverse-engineer its own label. Deleting
it and keeping the honest, lower score is the correct move, not a failure.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Named Limitation

This is an unbalanced panel: only 9 of 70 clients have 12+ months of history, and
`ga4_data_available` varies row by row. My month=2026-03 slice reflects whichever
clients happened to have tracking live that month -- it is not a representative
sample of all 104 clients, and any pattern found here needs re-confirming once I
pull the full multi-month panel with proper per-client history checks
(gsc_data_start / ga4_data_start).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.